# TT/MPS基礎 13 — TT Inner Product・Frobenius Norm・TT 間距離

## 今回の位置づけ

Notebook 12 では、既存の TT cores を dense tensor に復元せずに TT-rounding し、

$$
(r_1,\ldots,r_{d-1})
\longrightarrow
(\widetilde r_1,\ldots,\widetilde r_{d-1})
$$

と rank を削減しました。

次に必要なのは、

> **TT を dense tensor に戻さず、そのまま比較・評価する方法**

です。

今回は2本の実数値 TT、

$$
\mathcal A
=
\llbracket
A^{(1)},\ldots,A^{(d)}
\rrbracket,
\qquad
\mathcal B
=
\llbracket
B^{(1)},\ldots,B^{(d)}
\rrbracket
$$

について、

$$
\boxed{
\langle\mathcal A,\mathcal B\rangle
}
$$

$$
\boxed{
\|\mathcal A\|_F
}
$$

$$
\boxed{
\|\mathcal A-\mathcal B\|_F
}
$$

を、**dense tensor を構築せずに**計算します。

小規模な TT についてのみ `tt_to_dense` を使い、TT contraction の結果と dense 側の結果が一致することを確認します。

### 今回やること

1. TT の成分表示を復習する
2. dense tensor の Frobenius 内積を TT cores の縮約へ書き換える
3. left environment $E_k$ を定義する
4. environment 更新式を導く
5. `torch.einsum("ab,aic,bid->cd", ...)` と数式添字を対応させる
6. 最終 environment がスカラー内積になる理由を確認する
7. TT 内積 `tt_inner` を実装する
8. 自己内積から `tt_fro_norm` を実装する
9. 偏極恒等式から `tt_distance_sq` / `tt_distance` を実装する
10. dense reconstruction と `torch.testing.assert_close` で照合する
11. A/B で異なる TT-rank を使い、rank が一致しなくても縮約できることを確認する
12. 複素 MPS では bra 側の複素共役が必要になることを理解する
13. dense tensor を避ける計算量上の意味を整理する

### 今回はまだ扱わないもの

- TT-matrix
- TT-Linear
- Fashion-MNIST
- TT addition
- Hadamard product
- 複素数値 MPS の実装
- DMRG environment


## 1. TT の成分表示

$\mathcal A$ の第 $k$ core を、

$$
A^{(k)}
\in
\mathbb R^{
r_{k-1}^{A}
\times n_k
\times r_k^{A}
}
$$

とします。

境界 rank は、

$$
r_0^A=r_d^A=1
$$

です。

$\mathcal B$ も同様に、

$$
B^{(k)}
\in
\mathbb R^{
r_{k-1}^{B}
\times n_k
\times r_k^{B}
},
$$

$$
r_0^B=r_d^B=1
$$

とします。

physical dimension は A/B で一致させます。

$$
n_k^A=n_k^B=n_k.
$$

TT-rank は一致する必要はありません。

$$
r_k^A
\ne
r_k^B
$$

でも構いません。


## 2. 1つの dense 成分は TT core slice の行列積

物理添字、

$$
(i_1,\ldots,i_d)
$$

を固定すると、各 core の slice は、

$$
A^{(k)}[:,i_k,:]
\in
\mathbb R^{r_{k-1}^A\times r_k^A}
$$

という行列です。

したがって、

$$
\boxed{
\mathcal A_{i_1,\ldots,i_d}
=
A^{(1)}[:,i_1,:]\,
A^{(2)}[:,i_2,:]\,
\cdots
A^{(d)}[:,i_d,:]
}
$$

です。

境界 rank が、

$$
r_0^A=r_d^A=1
$$

なので、shape は、

$$
(1\times r_1^A)
(r_1^A\times r_2^A)
\cdots
(r_{d-1}^A\times1)
=
1\times1
$$

となり、最終結果はスカラーです。

添字を完全に書くと、

$$
\mathcal A_{i_1,\ldots,i_d}
=
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
A^{(1)}_{1,i_1,\alpha_1}
A^{(2)}_{\alpha_1,i_2,\alpha_2}
\cdots
A^{(d)}_{\alpha_{d-1},i_d,1}.
$$


## 3. Dense Tensor の Frobenius 内積

同じ physical shape を持つ2つの実数テンソルについて、

$$
\boxed{
\langle\mathcal A,\mathcal B\rangle
=
\sum_{i_1,\ldots,i_d}
\mathcal A_{i_1,\ldots,i_d}
\mathcal B_{i_1,\ldots,i_d}
}
$$

です。

TT の成分表示を代入すると、

$$
\begin{aligned}
\langle\mathcal A,\mathcal B\rangle
=
&
\sum_{i_1,\ldots,i_d}
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
\sum_{\beta_1,\ldots,\beta_{d-1}}
\\
&
\left(
A^{(1)}_{1,i_1,\alpha_1}
\cdots
A^{(d)}_{\alpha_{d-1},i_d,1}
\right)
\\
&\qquad\times
\left(
B^{(1)}_{1,i_1,\beta_1}
\cdots
B^{(d)}_{\beta_{d-1},i_d,1}
\right).
\end{aligned}
$$

この総和を physical index の全組み合わせについて直接評価すると、TT の利点を活かせません。

そこで、左から site ごとに縮約します。


## 4. Left Environment の定義

第 $k$ site までをすべて縮約した量を、

$$
E_k
$$

とします。

$E_k$ に残るのは、

- $\mathcal A$ の第 $k$ bond index $\alpha_k$
- $\mathcal B$ の第 $k$ bond index $\beta_k$

だけです。

したがって、

$$
\boxed{
E_k
\in
\mathbb R^{
r_k^A
\times
r_k^B
}
}
$$

です。

成分表示は、

$$
\begin{aligned}
E_k(\alpha_k,\beta_k)
:={}&
\sum_{i_1,\ldots,i_k}
\sum_{\alpha_1,\ldots,\alpha_{k-1}}
\sum_{\beta_1,\ldots,\beta_{k-1}}
\\
&
\left[
A^{(1)}_{1,i_1,\alpha_1}
\cdots
A^{(k)}_{\alpha_{k-1},i_k,\alpha_k}
\right]
\\
&\times
\left[
B^{(1)}_{1,i_1,\beta_1}
\cdots
B^{(k)}_{\beta_{k-1},i_k,\beta_k}
\right].
\end{aligned}
$$

処理済みの physical index と左 bond index はすべて和で消えています。


## 5. 初期 Environment

まだ1つも site を処理していない段階では、

$$
r_0^A=r_0^B=1
$$

です。

積の単位元として、

$$
\boxed{
E_0
=
[1]
\in
\mathbb R^{1\times1}
}
$$

とします。

PyTorch では、

```python
env = torch.ones(
    (1, 1),
    dtype=cores_a[0].dtype,
    device=cores_a[0].device,
)
```

です。

この `env` は入力 TT とは独立に新規作成します。

入力 cores は in-place 変更しません。


## 6. Environment の更新式

$E_{k-1}$ が既に計算できているとします。

shape は、

$$
E_{k-1}
\in
\mathbb R^{
r_{k-1}^A
\times
r_{k-1}^B
}
$$

です。

第 $k$ site の cores は、

$$
A^{(k)}
\in
\mathbb R^{
r_{k-1}^A
\times n_k
\times r_k^A
},
$$

$$
B^{(k)}
\in
\mathbb R^{
r_{k-1}^B
\times n_k
\times r_k^B
}
$$

です。

この3つを縮約すると、

$$
\boxed{
E_k(\alpha_k,\beta_k)
=
\sum_{\alpha_{k-1},\beta_{k-1},i_k}
E_{k-1}(\alpha_{k-1},\beta_{k-1})
A^{(k)}_{\alpha_{k-1},i_k,\alpha_k}
B^{(k)}_{\beta_{k-1},i_k,\beta_k}
}
$$

となります。

縮約する添字は、

$$
\alpha_{k-1},
\qquad
\beta_{k-1},
\qquad
i_k
$$

です。

残る添字は、

$$
\alpha_k,
\qquad
\beta_k
$$

なので、

$$
E_k
\in
\mathbb R^{r_k^A\times r_k^B}
$$

となります。


## 7. `torch.einsum` との完全な対応

更新式を、

```python
env = torch.einsum(
    "ab,aic,bid->cd",
    env,
    core_a,
    core_b,
)
```

と実装します。

対応は次です。

| `einsum` | 数式 | 意味 |
| --- | --- | --- |
| `a` | $\alpha_{k-1}$ | A の左 bond |
| `b` | $\beta_{k-1}$ | B の左 bond |
| `i` | $i_k$ | 共通 physical index |
| `c` | $\alpha_k$ | A の右 bond |
| `d` | $\beta_k$ | B の右 bond |

入力 shape は、

```text
env    : (a, b)
core_a : (a, i, c)
core_b : (b, i, d)
```

です。

`a`, `b`, `i` は出力側に書かれていないので総和されます。

出力は、

```text
(c, d)
```

すなわち、

$$
E_k
\in
\mathbb R^{r_k^A\times r_k^B}
$$

です。

この1行が今回の中心実装です。


## 8. 最終 Environment が Inner Product になる理由

最後の site $k=d$ まで処理すると、

$$
r_d^A=r_d^B=1
$$

なので、

$$
E_d
\in
\mathbb R^{1\times1}
$$

です。

左からすべての、

- physical index
- A の内部 bond
- B の内部 bond

を縮約済みなので、$E_d$ に残る自由な添字はありません。

したがって唯一の要素が、

$$
\boxed{
E_d(1,1)
=
\langle\mathcal A,\mathcal B\rangle
}
$$

です。

実装では、

```python
return env.squeeze()
```

として shape `()` の scalar tensor を返します。

`.item()` を返り値に使わない理由は、将来 autograd へつなぐときも計算グラフを保持するためです。


## 9. Environment Shape の具体例

検証では、

$$
(n_1,n_2,n_3)
=
(2,3,2)
$$

を使います。

A の rank 列を、

$$
(1,2,3,1)
$$

B の rank 列を、

$$
(1,3,2,1)
$$

とします。

core shapes は、

| site | A | B |
| ---: | --- | --- |
| 1 | $(1,2,2)$ | $(1,2,3)$ |
| 2 | $(2,3,3)$ | $(3,3,2)$ |
| 3 | $(3,2,1)$ | $(2,2,1)$ |

です。

environment は、

$$
\boxed{
(1,1)
\rightarrow
(2,3)
\rightarrow
(3,2)
\rightarrow
(1,1)
}
$$

と変化します。

A/B で異なる rank を使うことで、

> rank が偶然同じ場合だけ通る誤った実装

を避けます。


## 10. Frobenius Norm は Self Inner Product

実数テンソルでは、

$$
\|\mathcal A\|_F^2
=
\sum_{\boldsymbol i}
\mathcal A_{\boldsymbol i}^2
$$

です。

これは、

$$
\boxed{
\|\mathcal A\|_F^2
=
\langle\mathcal A,\mathcal A\rangle
}
$$

なので、

$$
\boxed{
\|\mathcal A\|_F
=
\sqrt{
\langle\mathcal A,\mathcal A\rangle
}
}
$$

です。

つまり `tt_inner` が実装できれば、新しい contraction アルゴリズムは不要です。

同じ TT を2回渡せばよいです。


## 11. TT 間距離は Difference Tensor を作らなくてもよい

Frobenius 距離は、

$$
\|\mathcal A-\mathcal B\|_F
$$

です。

二乗を展開すると、

$$
\begin{aligned}
\|\mathcal A-\mathcal B\|_F^2
&=
\langle
\mathcal A-\mathcal B,
\mathcal A-\mathcal B
\rangle
\\
&=
\langle\mathcal A,\mathcal A\rangle
+
\langle\mathcal B,\mathcal B\rangle
-
2\langle\mathcal A,\mathcal B\rangle.
\end{aligned}
$$

したがって、

$$
\boxed{
\|\mathcal A-\mathcal B\|_F^2
=
\|\mathcal A\|_F^2
+
\|\mathcal B\|_F^2
-
2\langle\mathcal A,\mathcal B\rangle
}
$$

です。

重要なのは、

> **$\mathcal A-\mathcal B$ という TT を実際に構築する必要がない**

ことです。

3回の TT inner product だけで距離二乗を計算できます。


## 12. 距離二乗の微小な負値

理論上、

$$
\|\mathcal A-\mathcal B\|_F^2
\ge0
$$

です。

しかし、

$$
\|\mathcal A\|_F^2
+
\|\mathcal B\|_F^2
-
2\langle\mathcal A,\mathcal B\rangle
$$

は近い数値どうしの引き算を含みます。

$\mathcal A$ と $\mathcal B$ が非常に近い場合、float64 でも丸め・桁落ちにより、

$$
-10^{-16}
$$

程度の微小な負値になる可能性があります。

そのため平方根の前に、

```python
distance_sq = distance_sq.clamp_min(0.0)
```

とします。

同様に自己内積から norm を取る場合も、

```python
norm_sq.clamp_min(0.0)
```

として安全に処理します。

### 注意

例えば、

$$
-10^{-3}
$$

のような大きな負値は、通常の丸め誤差とは考えにくいです。

その場合は、

- `einsum` 添字
- physical shape の整合性
- dtype
- 将来の複素数実装なら共役

などを疑います。


## 13. 複素 MPS との違い

今回は実数値 TT だけを実装します。

実数では、

$$
\langle\mathcal A,\mathcal B\rangle
=
\sum_{\boldsymbol i}
\mathcal A_{\boldsymbol i}
\mathcal B_{\boldsymbol i}.
$$

複素 MPS では bra 側を共役して、

$$
\boxed{
\langle\mathcal A,\mathcal B\rangle
=
\sum_{\boldsymbol i}
\overline{
\mathcal A_{\boldsymbol i}
}
\mathcal B_{\boldsymbol i}
}
$$

です。

したがって environment update では A 側 core に、

```python
core_a.conj()
```

を適用する必要があります。

今回は複素数 implementation までは行いません。


## 14. Dense を避ける計算量上の意味

physical dimensions を一様に、

$$
n_k\approx n
$$

TT-rank をおおむね、

$$
r_k^A\approx r_k^B\approx r
$$

とします。

dense tensor の要素数は、

$$
\boxed{
n^d
}
$$

で、次数 $d$ に対して指数的に増えます。

一方 TT の保存量は概ね、

$$
\boxed{
O(dnr^2)
}
$$

です。

left environment contraction も dense tensor の全要素を構築せず、site ごとの局所縮約として進みます。

典型的な rank / mode size の見積りでは、TT inner product はおおむね、

$$
\boxed{
O(dnr^3)
}
$$

程度で評価できます。

正確な計算量は A/B の各 site の rank に依存しますが、重要なのは、

$$
\prod_k n_k
$$

個の dense 要素を保持する必要がないことです。


## 15. PyTorch 設定


In [32]:
import torch

torch.manual_seed(0)

DTYPE = torch.float64
DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("dtype:", DTYPE)
print("device:", DEVICE)


PyTorch: 2.11.0+cu128
dtype: torch.float64
device: cpu


## 16. 既習の補助関数

ここでは前回までに使った、

- `validate_tt_cores`
- `tt_to_dense`
- `fro_norm`

を再利用します。

`tt_to_dense` は **小規模な正しさ検証専用**です。

本番の inner product / norm / distance の計算には使いません。


In [33]:
def validate_tt_cores(cores):
    assert isinstance(cores, (list, tuple))
    assert len(cores) >= 1

    dtype = cores[0].dtype
    device = cores[0].device

    for k, core in enumerate(cores):
        assert isinstance(core, torch.Tensor)
        assert core.ndim == 3
        assert all(dim >= 1 for dim in core.shape)
        assert core.dtype == dtype
        assert core.device == device

        if k > 0:
            assert cores[k - 1].shape[2] == core.shape[0]

    assert cores[0].shape[0] == 1
    assert cores[-1].shape[2] == 1


def tt_physical_shape(cores):
    validate_tt_cores(cores)
    return tuple(int(core.shape[1]) for core in cores)


def tt_ranks(cores):
    validate_tt_cores(cores)
    return (
        1,
        *[int(core.shape[2]) for core in cores[:-1]],
        1,
    )


def tt_to_dense(cores):
    """小規模な数値検証専用。"""
    validate_tt_cores(cores)

    dense = cores[0].squeeze(0)

    for core in cores[1:]:
        dense = torch.tensordot(
            dense,
            core,
            dims=([-1], [0]),
        )

    return dense.squeeze(-1)


def fro_norm(x):
    return torch.linalg.vector_norm(x.reshape(-1))


## 17. 検証用 Random TT を作る

A/B で physical shape は同じにし、TT-rank は意図的に変えます。

$$
(n_1,n_2,n_3)
=
(2,3,2)
$$

$$
\boldsymbol r^A
=
(1,2,3,1)
$$

$$
\boldsymbol r^B
=
(1,3,2,1)
$$

です。

`make_random_tt` は今回の数学の中心ではないため、検証データ生成用の補助関数として用意します。


In [34]:
def make_random_tt(
    physical_dims,
    ranks,
    *,
    dtype=DTYPE,
    device=DEVICE,
):
    assert len(ranks) == len(physical_dims) + 1
    assert ranks[0] == 1
    assert ranks[-1] == 1

    cores = []

    for k, n_k in enumerate(physical_dims):
        core = torch.randn(
            ranks[k],
            n_k,
            ranks[k + 1],
            dtype=dtype,
            device=device,
        )
        cores.append(core)

    validate_tt_cores(cores)
    return cores


physical_dims = (2, 3, 2)

ranks_a = (1, 2, 3, 1)
ranks_b = (1, 3, 2, 1)

cores_a = make_random_tt(
    physical_dims,
    ranks_a,
)

cores_b = make_random_tt(
    physical_dims,
    ranks_b,
)

print("A shapes:", [tuple(core.shape) for core in cores_a])
print("B shapes:", [tuple(core.shape) for core in cores_b])
print("A ranks:", tt_ranks(cores_a))
print("B ranks:", tt_ranks(cores_b))


A shapes: [(1, 2, 2), (2, 3, 3), (3, 2, 1)]
B shapes: [(1, 2, 3), (3, 3, 2), (2, 2, 1)]
A ranks: (1, 2, 3, 1)
B ranks: (1, 3, 2, 1)


## 18. 演習1 — 1 Site 分だけ Environment を更新する

### 目的

いきなり一般関数を書かず、第1 site だけを手で縮約します。

初期値は、

$$
E_0
\in
\mathbb R^{1\times1}.
$$

第1 cores は、

$$
A^{(1)}
\in
\mathbb R^{1\times2\times2},
$$

$$
B^{(1)}
\in
\mathbb R^{1\times2\times3}.
$$

したがって更新後は、

$$
E_1
\in
\mathbb R^{2\times3}
$$

になるはずです。

### TODO

1. `env0` を `(1, 1)` の ones tensor として作る
2. 
   ```python
   torch.einsum(
       "ab,aic,bid->cd",
       env0,
       cores_a[0],
       cores_b[0],
   )
   ```
   で `env1` を作る
3. `env1.shape == (2, 3)` を確認する

### 考えること

- `a`, `b`, `i` のどれが縮約されるか
- `c`, `d` がなぜ残るか
- A/B の右 rank が違うので、environment が長方形になること


In [4]:
# TODO 1:
# 第1 site だけ left environment を更新してください。
#
# 前提: 直前セルの cores_a / cores_b を使う
#   A^{(1)}: (1, 2, 2)  ← cores_a[0]
#   B^{(1)}: (1, 2, 3)  ← cores_b[0]
#
# 1. env0: shape (1, 1)
env0 = torch.ones(1, 1, dtype=cores_a[0].dtype, device=cores_a[0].device)

# 第1 site の core
A1 = cores_a[0]  # (1, 2, 2) = (r0^A, n1, r1^A)
B1 = cores_b[0]  # (1, 2, 3) = (r0^B, n1, r1^B)

print("env0:", tuple(env0.shape))
print("A1:", tuple(A1.shape))
print("B1:", tuple(B1.shape))
# 期待: env1.shape == (2, 3) = (r1^A, r1^B)
#
# 2. env1:
env1 = torch.einsum(
    "ab,aic,bid->cd",
    env0,
    A1,
    B1,
)

# 3. expected shape:
print("env1:", tuple(env1.shape))
assert tuple(env1.shape) == (2, 3)

env0: (1, 1)
A1: (1, 2, 2)
B1: (1, 2, 3)
env1: (2, 3)


## 19. 演習2 — Environment Shape を最後まで追う

### 目的

同じ更新を3 site すべてに適用し、

$$
(1,1)
\rightarrow
(2,3)
\rightarrow
(3,2)
\rightarrow
(1,1)
$$

になることを確認します。

### 実装する関数

```python
tt_inner_with_environment_shapes(
    cores_a,
    cores_b,
)
```

### 戻り値

- inner product の scalar tensor
- environment shapes の list

例えば shape 履歴は、

```python
[
    (1, 1),
    (2, 3),
    (3, 2),
    (1, 1),
]
```

です。

### TODO

1. A/B の order が同じか確認する
2. physical shape が同じか確認する
3. dtype / device が同じか確認する
4. `env = ones((1, 1))` から開始
5. site ごとに `einsum`
6. 更新後 shape を保存
7. 最後の environment が `(1, 1)` か確認
8. `env.squeeze()` と shape 履歴を返す

### 考えること

- A/B の rank が異なっても問題ない理由
- physical dimension だけは一致する必要がある理由
- なぜ最終 environment は必ず `(1, 1)` になるのか


In [36]:
# TODO 2:
# tt_inner_with_environment_shapes を実装してください。
#
def tt_inner_with_environment_shapes(
    cores_a: list[torch.Tensor],
    cores_b: list[torch.Tensor],
) -> tuple[torch.Tensor, list[tuple[int, int]]]:
    """TT 内積と left environment の shape 履歴を返す。

    引数:
        cores_a: TT A の core 列
            [A^{(1)}, ..., A^{(d)}],
            A^{(k)}.shape = (r_{k-1}^A, n_k, r_k^A),
            r_0^A = r_d^A = 1
        cores_b: TT B の core 列
            [B^{(1)}, ..., B^{(d)}],
            B^{(k)}.shape = (r_{k-1}^B, n_k, r_k^B),
            r_0^B = r_d^B = 1
            （physical shape (n_1,...,n_d) は A と同じであること）

    戻り値:
        inner: スカラー Tensor（shape ()）
            ⟨A, B⟩ = E_d.squeeze()
        env_shapes: list[tuple[int, int]]
            E_0, E_1, ..., E_d の shape 履歴
            例: [(1, 1), (2, 3), (3, 2), (1, 1)]
    """
    # 1. A/B の order（site 数）が同じか
    validate_tt_cores(cores_a)
    validate_tt_cores(cores_b)
    assert len(cores_a) == len(cores_b), (
        f"order が一致しません: d_A={len(cores_a)}, d_B={len(cores_b)}"
    )

    # 2. physical shape が同じか
    phys_a = tt_physical_shape(cores_a)
    phys_b = tt_physical_shape(cores_b)
    assert phys_a == phys_b, (
        f"physical shape が一致しません: A={phys_a}, B={phys_b}"
    )

    # 3. dtype / device が同じか
    assert cores_a[0].dtype == cores_b[0].dtype, (
        f"dtype が一致しません: A={cores_a[0].dtype}, B={cores_b[0].dtype}"
    )
    assert cores_a[0].device == cores_b[0].device, (
        f"device が一致しません: A={cores_a[0].device}, B={cores_b[0].device}"
    )

    env = torch.ones(1, 1, dtype=cores_a[0].dtype, device=cores_a[0].device)
    env_shapes = [tuple(env.shape)]  # E_0

    # 5. site ごとに einsum
    # 6. 更新後 shape を保存
    for k in range(len(cores_a)):
        env = torch.einsum(
            "ab,aic,bid->cd",
            env,
            cores_a[k],
            cores_b[k],
        )
        env_shapes.append(tuple(env.shape))

    # 7. 最後の environment が (1, 1) か確認
    assert tuple(env.shape) == (1, 1), f"最終 env shape が (1, 1) ではありません: {tuple(env.shape)}"

    # 8. env.squeeze() と shape 履歴を返す
    return env.squeeze(), env_shapes


## 20. 演習3 — `tt_inner`

### 目的

environment shape の学習用記録を外し、本番用の inner product 関数を作ります。

### API

```python
inner = tt_inner(
    cores_a,
    cores_b,
)
```

返り値は、

```python
inner.shape == torch.Size([])
```

となる scalar tensor にします。

### TODO

1. 入力検証
2. physical shape の一致確認
3. dtype / device の一致確認
4. `(1, 1)` environment 初期化
5. 全 site を `einsum`
6. 最終 `(1, 1)` environment を確認
7. `.squeeze()` して返す

### 重要

返り値に `.item()` は使いません。

将来 TT cores を `nn.Parameter` にしたときも autograd graph を保持できる形にします。


In [37]:
# TODO 3:
# tt_inner を実装してください。
#
def tt_inner(cores_a, cores_b):
    # 1. A/B の order（site 数）が同じか
    validate_tt_cores(cores_a)
    validate_tt_cores(cores_b)
    assert len(cores_a) == len(cores_b), (
        f"order が一致しません: d_A={len(cores_a)}, d_B={len(cores_b)}"
    )

    # 2. physical shape が同じか
    phys_a = tt_physical_shape(cores_a)
    phys_b = tt_physical_shape(cores_b)
    assert phys_a == phys_b, (
        f"physical shape が一致しません: A={phys_a}, B={phys_b}"
    )

    # 3. dtype / device が同じか
    assert cores_a[0].dtype == cores_b[0].dtype, (
        f"dtype が一致しません: A={cores_a[0].dtype}, B={cores_b[0].dtype}"
    )
    assert cores_a[0].device == cores_b[0].device, (
        f"device が一致しません: A={cores_a[0].device}, B={cores_b[0].device}"
    )
    env = torch.ones(1, 1, dtype=cores_a[0].dtype, device=cores_a[0].device)
    for k in range(len(cores_a)):
        env = torch.einsum(
            "ab,aic,bid->cd",
            env,
            cores_a[k],
            cores_b[k],
        )
    # 7. 最後の environment が (1, 1) か確認
    assert tuple(env.shape) == (1, 1), f"最終 env shape が (1, 1) ではありません: {tuple(env.shape)}"
    return env.squeeze()

## 21. 演習4 — `tt_fro_norm`

### 目的

自己内積から Frobenius norm を計算します。

$$
\|\mathcal A\|_F
=
\sqrt{
\langle\mathcal A,\mathcal A\rangle
}.
$$

### TODO

1. `tt_inner(cores, cores)` で `norm_sq` を作る
2. `norm_sq.clamp_min(0.0)` を適用する
3. `torch.sqrt` を取る
4. scalar tensor を返す

### 考えること

- なぜ norm 専用の environment contraction を新しく書く必要がないのか
- 理論上 `norm_sq >= 0` なのに clamp を入れる理由


In [38]:
# TODO 4:
# tt_fro_norm を実装してください。
#
def tt_fro_norm(cores):
    norm_sq=tt_inner(cores,cores)
    safe_norm_sq=norm_sq.clamp_min(0.0)
    return torch.sqrt(safe_norm_sq)


## 22. 演習5 — `tt_distance_sq`

### 目的

差の TT を明示的に構築せず、

$$
\boxed{
\|\mathcal A-\mathcal B\|_F^2
=
\langle\mathcal A,\mathcal A\rangle
+
\langle\mathcal B,\mathcal B\rangle
-
2\langle\mathcal A,\mathcal B\rangle
}
$$

から距離二乗を計算します。

### TODO

1. `aa = tt_inner(cores_a, cores_a)`
2. `bb = tt_inner(cores_b, cores_b)`
3. `ab = tt_inner(cores_a, cores_b)`
4. 上式から `distance_sq` を作る
5. `clamp_min(0.0)` を適用
6. scalar tensor を返す

### 考えること

- `A-B` の TT core を作らなくてよいこと
- なぜ A/B の TT-rank が異なっても距離を計算できるのか
- clamp 前の値が大きく負なら何を疑うべきか


In [39]:
# TODO 5:
# tt_distance_sq を実装してください。
#
def tt_distance_sq(cores_a, cores_b):
    aa = tt_inner(cores_a, cores_a)
    bb = tt_inner(cores_b, cores_b)
    ab = tt_inner(cores_a, cores_b)
    distance_sq=aa + bb - 2*ab
    safe_distance_sq=distance_sq.clamp_min(0.0)
    return safe_distance_sq

## 23. 演習6 — `tt_distance`

### 目的

距離二乗から、

$$
\boxed{
\|\mathcal A-\mathcal B\|_F
=
\sqrt{
\|\mathcal A-\mathcal B\|_F^2
}
}
$$

を計算します。

### TODO

`tt_distance_sq` を再利用して `tt_distance` を実装してください。

新しい inner contraction を書き直す必要はありません。


In [9]:
# TODO 6:
# tt_distance を実装してください。
#
def tt_distance(cores_a, cores_b):
    distance_sq = tt_distance_sq(cores_a, cores_b)
    return torch.sqrt(distance_sq)


## 24. 演習7 — Environment Shape を数値確認する

### 目的

演習2の関数を実行し、

$$
(1,1)
\rightarrow
(2,3)
\rightarrow
(3,2)
\rightarrow
(1,1)
$$

を assert します。

### TODO

1. `tt_inner_with_environment_shapes` を実行
2. shape 履歴を表示
3. 期待する list と一致することを assert
4. 最終 inner が shape `()` であることを確認する


In [10]:
# TODO 7:
expected_environment_shapes = [
    (1, 1),
    (2, 3),
    (3, 2),
    (1, 1),
]
#
# environment shape の遷移を確認してください。
#
# 1. 実行（直前セルの cores_a / cores_b を使う）
inner, env_shapes = tt_inner_with_environment_shapes(cores_a, cores_b)

# 2. 表示
print("env_shapes:", env_shapes)
print("inner:", inner)
print("inner.shape:", tuple(inner.shape))

# 3. assert: env_shapes == expected_environment_shapes
assert env_shapes == expected_environment_shapes,f"shapes が一致しません: env_shapes={env_shapes}, expected_environment_shapes{expected_environment_shapes}"

# 4. assert: inner.shape == ()
assert inner.shape == (), (
    f"inner must be a scalar tensor with shape (), "
    f"but got {tuple(inner.shape)}."
)



env_shapes: [(1, 1), (2, 3), (3, 2), (1, 1)]
inner: tensor(14.2433, dtype=torch.float64)
inner.shape: ()


## 25. Dense 側の正解値を作る

ここからのみ、小規模検証用 `tt_to_dense` を使います。

$$
\mathcal A_{\mathrm{dense}}
=
\operatorname{tt\_to\_dense}(\mathcal A),
$$

$$
\mathcal B_{\mathrm{dense}}
=
\operatorname{tt\_to\_dense}(\mathcal B).
$$

dense 側では定義どおり直接、

### Inner Product

$$
\sum_{\boldsymbol i}
A_{\boldsymbol i}B_{\boldsymbol i}
$$

### Norm

$$
\sqrt{
\sum_{\boldsymbol i}
A_{\boldsymbol i}^2
}
$$

### Distance

$$
\sqrt{
\sum_{\boldsymbol i}
(A_{\boldsymbol i}-B_{\boldsymbol i})^2
}
$$

を計算します。

これは本番アルゴリズムではなく、TT contraction 実装の **正解生成器**として使います。


In [42]:
dense_a = tt_to_dense(cores_a)
dense_b = tt_to_dense(cores_b)

print("dense A shape:", tuple(dense_a.shape))
print("dense B shape:", tuple(dense_b.shape))


dense A shape: (2, 3, 2)
dense B shape: (2, 3, 2)


## 26. 演習8 — TT Inner Product と Dense Inner Product

### 目的

$$
\boxed{
\operatorname{tt\_inner}(\mathcal A,\mathcal B)
=
\sum_{\boldsymbol i}
A_{\boldsymbol i}B_{\boldsymbol i}
}
$$

を数値確認します。

### Dense 側

```python
inner_dense = torch.sum(
    dense_a * dense_b
)
```

または、

```python
torch.dot(
    dense_a.reshape(-1),
    dense_b.reshape(-1),
)
```

でも同じです。

### TODO

1. TT 側 inner product
2. dense 側 inner product
3. 絶対差を表示
4. 
   ```python
   torch.testing.assert_close(
       ...,
       atol=1e-12,
       rtol=1e-12,
   )
   ```
   で確認する


In [43]:
# TODO 8:
# TT inner product と dense inner product を比較してください。
#
# 1. TT 側
inner_tt = tt_inner(cores_a, cores_b)

# 2. dense 側
dense_a = tt_to_dense(cores_a)
dense_b = tt_to_dense(cores_b)
inner_dense = torch.sum(dense_a * dense_b)

# 3. 絶対差
print("inner_tt:", inner_tt)
print("inner_dense:", inner_dense)
print("|tt - dense|:", (inner_tt - inner_dense).abs().item())

# 4. 数値一致を確認
torch.testing.assert_close(
    inner_tt,
    inner_dense,
    atol=1e-12,
    rtol=1e-12,
)
print("OK: tt_inner ≈ dense inner")


inner_tt: tensor(14.2433, dtype=torch.float64)
inner_dense: tensor(14.2433, dtype=torch.float64)
|tt - dense|: 1.7763568394002505e-15
OK: tt_inner ≈ dense inner


## 27. 演習9 — Frobenius Norm を Dense と照合する

### 目的

A/B それぞれについて、

$$
\boxed{
\operatorname{tt\_fro\_norm}(\mathcal A)
=
\|\mathcal A_{\mathrm{dense}}\|_F
}
$$

を確認します。

dense 側は、

```python
torch.linalg.vector_norm(
    dense_a.reshape(-1)
)
```

です。

### TODO

- A の TT norm / dense norm
- B の TT norm / dense norm
- 各絶対差
- `torch.testing.assert_close`


In [44]:
# TODO 9:
# A/B の Frobenius norm を dense 側と比較してください。
#
dense_a = tt_to_dense(cores_a)
dense_b = tt_to_dense(cores_b)

# --- A ---
norm_a_tt = tt_fro_norm(cores_a)
norm_a_dense = torch.linalg.vector_norm(dense_a.reshape(-1))
print("A tt / dense:", norm_a_tt.item(), "/", norm_a_dense.item())
print("|A tt - dense|:", (norm_a_tt - norm_a_dense).abs().item())
torch.testing.assert_close(norm_a_tt, norm_a_dense, atol=1e-12, rtol=1e-12)

# --- B ---
norm_b_tt = tt_fro_norm(cores_b)
norm_b_dense = torch.linalg.vector_norm(dense_b.reshape(-1))
print("B tt / dense:", norm_b_tt.item(), "/", norm_b_dense.item())
print("|B tt - dense|:", (norm_b_tt - norm_b_dense).abs().item())
torch.testing.assert_close(norm_b_tt, norm_b_dense, atol=1e-12, rtol=1e-12)

print("OK: tt_fro_norm ≈ dense Frobenius norm")


A tt / dense: 7.7673103257282525 / 7.767310325728252
|A tt - dense|: 8.881784197001252e-16
B tt / dense: 5.70851544682925 / 5.70851544682925
|B tt - dense|: 0.0
OK: tt_fro_norm ≈ dense Frobenius norm


## 28. 演習10 — Distance Squared と Distance を Dense と照合する

### 目的

$$
\boxed{
\operatorname{tt\_distance\_sq}
=
\sum_{\boldsymbol i}
(A_{\boldsymbol i}-B_{\boldsymbol i})^2
}
$$

と、

$$
\boxed{
\operatorname{tt\_distance}
=
\|\mathcal A_{\mathrm{dense}}-\mathcal B_{\mathrm{dense}}\|_F
}
$$

を確認します。

### Dense 側

```python
distance_sq_dense = torch.sum(
    (dense_a - dense_b) ** 2
)

distance_dense = torch.linalg.vector_norm(
    (dense_a - dense_b).reshape(-1)
)
```

### TODO

1. TT distance squared
2. dense distance squared
3. TT distance
4. dense distance
5. 両方 `assert_close`


In [14]:
# TODO 10:
# TT-only の distance squared / distance を dense 側と比較する。

# TT 側
distance_sq_tt = tt_distance_sq(cores_a, cores_b)
distance_tt = tt_distance(cores_a, cores_b)

# dense 側
distance_sq_dense = torch.sum((dense_a - dense_b) ** 2)
distance_dense = torch.linalg.vector_norm(dense_a - dense_b)

# 表示
print("distance_sq_tt:", distance_sq_tt)
print("distance_sq_dense:", distance_sq_dense)
print(
    "|distance_sq_tt - distance_sq_dense|:",
    (distance_sq_tt - distance_sq_dense).abs().item(),
)
print()
print("distance_tt:", distance_tt)
print("distance_dense:", distance_dense)
print(
    "|distance_tt - distance_dense|:",
    (distance_tt - distance_dense).abs().item(),
)

# 数値照合
torch.testing.assert_close(
    distance_sq_tt,
    distance_sq_dense,
    atol=1e-12,
    rtol=1e-12,
)

torch.testing.assert_close(
    distance_tt,
    distance_dense,
    atol=1e-12,
    rtol=1e-12,
)

print("OK: tt_distance_sq ≈ dense distance squared")
print("OK: tt_distance ≈ dense Frobenius distance")


distance_sq_tt: tensor(64.4317, dtype=torch.float64)
distance_sq_dense: tensor(64.4317, dtype=torch.float64)
|distance_sq_tt - distance_sq_dense|: 0.0

distance_tt: tensor(8.0269, dtype=torch.float64)
distance_dense: tensor(8.0269, dtype=torch.float64)
|distance_tt - distance_dense|: 0.0
OK: tt_distance_sq ≈ dense distance squared
OK: tt_distance ≈ dense Frobenius distance


## 29. 演習11 — $\|\mathcal A-\mathcal A\|_F=0$

### 目的

最も基本的な sanity check として、

$$
\boxed{
\|\mathcal A-\mathcal A\|_F=0
}
$$

を確認します。

理論上は厳密に0です。

浮動小数点計算では距離二乗の展開に桁落ちが入る可能性があるため、`clamp_min(0.0)` が安全策になります。

### TODO

```python
torch.testing.assert_close(
    tt_distance(cores_a, cores_a),
    torch.zeros(
        (),
        dtype=DTYPE,
        device=DEVICE,
    ),
    atol=1e-12,
    rtol=1e-12,
)
```

と同等の確認を行ってください。


In [46]:
# TODO 11:
# ||A - A||_F = 0 を確認してください。
#
dist_aa = tt_distance(cores_a, cores_a)
print("||A - A||_F:", dist_aa.item())

torch.testing.assert_close(
    dist_aa,
    torch.zeros((), dtype=DTYPE, device=DEVICE),
    atol=1e-12,
    rtol=1e-12,
)
print("OK: ||A - A||_F ≈ 0")


||A - A||_F: 0.0
OK: ||A - A||_F ≈ 0


## 30. 演習12 — 入力検証を壊してみる

### 目的

正しい数値だけでなく、誤った入力を静かに受け入れないことも確認します。

例えば B のある core だけ physical dimension を変えた場合、

$$
n_k^A
\ne
n_k^B
$$

となり、inner product は定義できません。

### TODO

次のようなエラーケースを1つずつ作り、`tt_inner` が拒否することを確認してください。

- A/B の order が異なる
- physical dimension が異なる
- dtype が異なる
- device が異なる場合
- TT 内部で隣接 bond shape が壊れている

### 考えること

- TT-rank が異なること自体はエラーではない
- physical shape が異なることはエラーである


In [48]:
# TODO 12:
# 不正入力に対する validation を確認してください。
#
def expect_fail(label, fn):
    try:
        fn()
    except AssertionError as e:
        print(f"OK [{label}]: {e}")
        return
    raise AssertionError(f"NG [{label}]: エラーになるはずなのに通りました")


# --- 1. A/B の order が異なる ---
# cores_a は3階TT。
# 比較対象として、境界rankが正しい正常な2階TTを作る。
cores_b_short = make_random_tt(
    physical_dims=(2, 3),
    ranks=(1, 3, 1),
    dtype=DTYPE,
    device=DEVICE,
)

expect_fail(
    "order mismatch",
    lambda: tt_inner(cores_a, cores_b_short),
)

# --- 2. physical dimension が異なる ---
cores_b_phys = [G.clone() for G in cores_b]
# site 1 の physical を 3 → 4 に壊す（左 bond は維持）
r_left, _, r_right = cores_b_phys[1].shape
cores_b_phys[1] = torch.randn(r_left, 4, r_right, dtype=DTYPE, device=DEVICE)
expect_fail("physical mismatch", lambda: tt_inner(cores_a, cores_b_phys))

# --- 3. dtype が異なる ---
# ノートの DTYPE は float64 なので、わざと float32 にする
cores_b_dtype = [G.to(torch.float32) for G in cores_b]
expect_fail("dtype mismatch", lambda: tt_inner(cores_a, cores_b_dtype))

# --- 4. device が異なる（CUDA があるときだけ） ---
if torch.cuda.is_available():
    cores_b_cuda = [G.to("cuda") for G in cores_b]
    expect_fail("device mismatch", lambda: tt_inner(cores_a, cores_b_cuda))
else:
    print("SKIP [device mismatch]: CUDA なし")

# --- 5. TT 内部で隣接 bond shape が壊れている ---
cores_broken = [G.clone() for G in cores_a]
# A の site0 右 bond と site1 左 bond を不一致にする
r0, n0, _ = cores_broken[0].shape
cores_broken[0] = torch.randn(r0, n0, 5, dtype=DTYPE, device=DEVICE)
expect_fail("broken bond", lambda: tt_inner(cores_broken, cores_b))

# --- 参考: TT-rank が違うこと自体はエラーではない ---
inner_ok = tt_inner(cores_a, cores_b)
print("OK [different TT-ranks allowed]:", inner_ok.item())


OK [order mismatch]: order が一致しません: d_A=3, d_B=2
OK [physical mismatch]: physical shape が一致しません: A=(2, 3, 2), B=(2, 4, 2)
OK [dtype mismatch]: dtype が一致しません: A=torch.float64, B=torch.float32
OK [device mismatch]: device が一致しません: A=cpu, B=cuda:0
OK [broken bond]: 
OK [different TT-ranks allowed]: 14.243284453983867


## 31. 理論記号とコード変数の対応

| 理論 | コード | 意味 |
| --- | --- | --- |
| $\mathcal A$ | `cores_a` | 1本目の TT |
| $\mathcal B$ | `cores_b` | 2本目の TT |
| $A^{(k)}$ | `core_a` | A の第 $k$ core |
| $B^{(k)}$ | `core_b` | B の第 $k$ core |
| $E_k$ | `env` | 第 $k$ site までの left environment |
| $\alpha_{k-1}$ | `a` | A の左 bond |
| $\beta_{k-1}$ | `b` | B の左 bond |
| $i_k$ | `i` | physical index |
| $\alpha_k$ | `c` | A の右 bond |
| $\beta_k$ | `d` | B の右 bond |
| $\langle\mathcal A,\mathcal B\rangle$ | `tt_inner(...)` | TT inner product |
| $\|\mathcal A\|_F$ | `tt_fro_norm(...)` | TT Frobenius norm |
| $\|\mathcal A-\mathcal B\|_F^2$ | `tt_distance_sq(...)` | TT distance squared |
| $\|\mathcal A-\mathcal B\|_F$ | `tt_distance(...)` | TT distance |

今回もっとも重要な対応は、

```text
ab,aic,bid->cd
```

と、

$$
(\alpha_{k-1},\beta_{k-1}),
\quad
(\alpha_{k-1},i_k,\alpha_k),
\quad
(\beta_{k-1},i_k,\beta_k)
\longrightarrow
(\alpha_k,\beta_k)
$$

です。


## 32. 今回のまとめ

### TT Inner Product

left environment を、

$$
E_0=[1]
$$

から開始し、

$$
\boxed{
E_k(\alpha_k,\beta_k)
=
\sum_{\alpha_{k-1},\beta_{k-1},i_k}
E_{k-1}(\alpha_{k-1},\beta_{k-1})
A^{(k)}_{\alpha_{k-1},i_k,\alpha_k}
B^{(k)}_{\beta_{k-1},i_k,\beta_k}
}
$$

で更新します。

実装は、

```python
env = torch.einsum(
    "ab,aic,bid->cd",
    env,
    core_a,
    core_b,
)
```

です。

境界 rank が1なので、

$$
E_d\in\mathbb R^{1\times1}
$$

となり、

$$
\boxed{
E_d(1,1)
=
\langle\mathcal A,\mathcal B\rangle
}
$$

です。

### Frobenius Norm

$$
\boxed{
\|\mathcal A\|_F
=
\sqrt{
\langle\mathcal A,\mathcal A\rangle
}
}
$$

です。

### TT 間距離

$$
\boxed{
\|\mathcal A-\mathcal B\|_F^2
=
\langle\mathcal A,\mathcal A\rangle
+
\langle\mathcal B,\mathcal B\rangle
-
2\langle\mathcal A,\mathcal B\rangle
}
$$

なので、差の TT を作らず計算できます。

### Dense Reconstruction の位置づけ

dense reconstruction は本番計算には使いません。

今回の小規模問題では、

> TT contraction が定義どおりの inner / norm / distance を計算していることを確認する正解生成器

としてのみ使います。

この Notebook を完了すると、TT-rounding 前後の誤差や2つの TT 表現の差を、dense tensor を作らずに測れる状態になります。

次は TT-matrix と TT-Linear へ進み、NN の Linear weight を TT 表現で扱う段階へ接続できます。
